# 🔢 Módulo 3: NumPy

Arrays, indexado, cálculo vectorizado y broadcasting: el motor numérico debajo de pandas.

**PandasQuest** · versión web interactiva: [https://rosalesluciano.github.io/pandas-quest/](https://rosalesluciano.github.io/pandas-quest/#/m/m03)

**Cómo funciona:**
1. Ejecuta la celda ⚙️ de abajo (carga las tablas y el corrector).
2. En cada ejercicio escribe tu código y guarda la respuesta en `resultado`.
3. Ejecuta la celda: `comprobar()` te dirá si está bien. 🎯

¿Atascado? `ver_solucion("id")` muestra la solución (¡pero inténtalo antes! 😉)

In [ ]:
#@title ⚙️ 1) Ejecuta esta celda primero (carga datos y corrector) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    return False, f"Tu resultado es {u!r}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usa & (y), | (o) y pon cada condición entre paréntesis."),
    ("KeyError", "Revisa que el nombre de la columna esté bien escrito (mayúsculas, tildes, espacios)."),
    ("NameError", "Estás usando una variable que no existe. ¿Un error de tipeo?"),
    ("SyntaxError", "Hay un error de sintaxis: revisa paréntesis, comillas y que la línea `resultado = ...` esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este objeto. Revisa el nombre."),
    ("TypeError", "Algún tipo de dato no encaja (¿texto mezclado con números?)."),
    ("IndentationError", "Revisa la sangría (espacios al inicio de la línea)."),
    ("MergeError", "Revisa las columnas que usas para unir (on / left_on / right_on)."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\ntemperaturas = np.array([21.5, 23.0, 19.8, 25.1, 27.3, 22.4, 18.9])   # una semana, en °C\nnotas = np.array([[7, 8, 6],\n                  [9, 5, 8],\n                  [6, 7, 9],\n                  [8, 9, 10],\n                  [5, 6, 7]])                                          # 5 alumnos x 3 exámenes\nprecios = productos["precio"].to_numpy()\n'
_SOL_B64 = 'eyJucDAxIjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBucC5hcnJheShbNSwgMTAsIDE1LCAyMF0pIiwgImNoZWNrIjoge30sICJ4cCI6IDEwfSwgIm5wMDIiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IG5wLmFyYW5nZSgwLCA1MCwgNSkiLCAiY2hlY2siOiB7fSwgInhwIjogMTB9LCAibnAwMyI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gbm90YXMuc2hhcGUiLCAiY2hlY2siOiB7fSwgInhwIjogMTB9LCAibnAwNCI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gbnAuYXJhbmdlKDEyKS5yZXNoYXBlKDMsIDQpIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgIm5wMDUiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IG5vdGFzWzosIDFdIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgIm5wMDYiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IG5vdGFzWzozLCAxOl0iLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAibnAwNyI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gdGVtcGVyYXR1cmFzW3RlbXBlcmF0dXJhcyA+IDIyXSIsICJjaGVjayI6IHt9LCAieHAiOiAyMH0sICJucDA4IjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSB0ZW1wZXJhdHVyYXMgKiA5IC8gNSArIDMyIiwgImNoZWNrIjoge30sICJ4cCI6IDEwfSwgIm5wMDkiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IG5vdGFzLm1lYW4oYXhpcz0xKSIsICJjaGVjayI6IHt9LCAieHAiOiAzNX0sICJucDEwIjogeyJzb2x1dGlvbiI6ICJib251cyA9IG5wLmFycmF5KFsxLCAwLCAwLjVdKVxucmVzdWx0YWRvID0gbnAubWluaW11bShub3RhcyArIGJvbnVzLCAxMCkiLCAiY2hlY2siOiB7fSwgInhwIjogMzV9LCAibnAxMSI6IHsic29sdXRpb24iOiAicHJvbWVkaW9zID0gbm90YXMubWVhbihheGlzPTEpXG5yZXN1bHRhZG8gPSBucC53aGVyZShwcm9tZWRpb3MgPj0gNywgJ2Fwcm9iYWRvJywgJ3JlY3VwZXJhJykiLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAibnAxMiI6IHsic29sdXRpb24iOiAicm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApXG5yZXN1bHRhZG8gPSBybmcuaW50ZWdlcnMoMSwgNywgc2l6ZT0xMCkiLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAibnAxMyI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gKHRlbXBlcmF0dXJhcyAtIHRlbXBlcmF0dXJhcy5tZWFuKCkpIC8gdGVtcGVyYXR1cmFzLnN0ZCgpIiwgImNoZWNrIjoge30sICJ4cCI6IDM1fSwgIm5wMTQiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IG5wLnBlcmNlbnRpbGUocHJlY2lvcywgNzUpIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgIm5wMTUiOiB7InNvbHV0aW9uIjogInBlc29zID0gbnAuYXJyYXkoWzAuMywgMC4zLCAwLjRdKVxucmVzdWx0YWRvID0gbm90YXMgQCBwZXNvcyIsICJjaGVjayI6IHt9LCAieHAiOiA1MH19'
_TABLAS = ['temperaturas', 'notas', 'precios']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todas las tablas limpias (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    if ok:
        from IPython.display import HTML, display
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">🎉 ¡Correcto! +' + str(info["xp"]) + ' XP · Sigue así 🐼</div>'))
    else:
        from IPython.display import HTML, display
        import html as _h
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">❌ Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_solucion(ej):
    print(_SOL[ej]["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 Crear arrays

## 1. Tu primer array
`Básico` · **+10 XP**

**NumPy** (*Numerical Python*) es la librería de cálculo numérico de Python y el **motor que hay debajo de pandas**: cada columna de un DataFrame guarda sus datos en un array de NumPy.

Su estructura básica es el **array** (`ndarray`): una colección de valores **del mismo tipo**, guardados de forma compacta y muy rápidos de operar.

```python
import numpy as np    # "np" es el alias estándar

a = np.array([3, 7, 1])
a * 2          # array([ 6, 14,  2])  <- sin bucles
a.dtype        # dtype('int64')
```

**¿Por qué no usar una lista?** Porque `[3, 7, 1] * 2` en una lista **repite** la lista (`[3, 7, 1, 3, 7, 1]`), mientras que el array **multiplica cada número**. Además, NumPy es entre 10 y 100 veces más rápido.

> 💼 **En el trabajo:** Machine Learning, imágenes, audio y simulaciones funcionan sobre arrays de NumPy.

### 🎯 Tu misión
Crea un array de NumPy con los valores `5, 10, 15, 20` y guárdalo en `resultado`.

<details><summary>💡 Pista</summary>

`np.array([5, 10, 15, 20])`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

# numpy ya está importado como np
resultado = 

comprobar("np01")

## 2. Rangos con arange y linspace
`Básico` · **+10 XP**

Para no escribir los números a mano, NumPy tiene funciones que **generan secuencias**:

- `np.arange(inicio, fin, paso)`: como `range()`; el **fin no se incluye**.
- `np.linspace(inicio, fin, cantidad)`: *cantidad* números equiespaciados, con el **fin incluido**.
- `np.zeros(n)` / `np.ones(n)`: arrays de ceros o de unos.

```python
np.arange(0, 10, 2)      # [0 2 4 6 8]
np.linspace(0, 1, 5)     # [0.   0.25 0.5  0.75 1.  ]
np.zeros(3)              # [0. 0. 0.]
```

> 💡 `linspace` es muy útil para dibujar curvas: generas, por ejemplo, 100 puntos entre 0 y 10.

### 🎯 Tu misión
Crea con `np.arange` el array `0, 5, 10, ..., 45` (de 0 a 45, de 5 en 5). Guárdalo en `resultado`.

<details><summary>💡 Pista</summary>

`np.arange(0, 50, 5): el 50 no se incluye`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np02")

## 3. Dimensiones y forma
`Básico` · **+10 XP**

Un array puede tener varias **dimensiones**:
- 1D: un vector, como `temperaturas`;
- 2D: una **matriz** con filas y columnas, como `notas`;
- 3D o más: por ejemplo una imagen a color (alto × ancho × 3 colores).

Atributos clave:
- `a.ndim`: cantidad de dimensiones;
- `a.shape`: tamaño de cada dimensión (`(filas, columnas)` en 2D);
- `a.size`: cantidad total de elementos;
- `a.dtype`: tipo de los datos.

En este módulo tienes cargados `temperaturas` (7 días), `notas` (5 alumnos × 3 exámenes) y `precios` (los precios de `productos`).

### 🎯 Tu misión
Guarda en `resultado` la **forma** (`shape`) de la matriz `notas`.

<details><summary>💡 Pista</summary>

`notas.shape`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

print(notas)
resultado = 

comprobar("np03")

## 4. Cambiar de forma
`Intermedio` · **+20 XP**

`reshape(filas, columnas)` reorganiza los mismos datos con otra forma. La cantidad total de elementos debe coincidir: 12 elementos pueden ser 3×4, 4×3, 2×6…

```python
np.arange(6).reshape(2, 3)
# [[0 1 2]
#  [3 4 5]]
```

Si pones `-1` en una dimensión, NumPy la **calcula por ti**: `reshape(3, -1)`.

> 💼 **En el trabajo:** scikit-learn exige que las variables de entrada sean 2D. `x.reshape(-1, 1)` convierte un vector en una columna, y lo verás muy a menudo.

### 🎯 Tu misión
Crea un array con los números del **0 al 11** y dale forma de **3 filas × 4 columnas**.

<details><summary>💡 Pista</summary>

`np.arange(12).reshape(3, 4)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np04")

---
# 📌 Indexar y recortar

## 5. Acceder a elementos
`Intermedio` · **+20 XP**

En un array 2D se indica `[fila, columna]`, empezando en 0:

```python
notas[0, 0]     # primera nota del primer alumno
notas[1]        # toda la fila 1 (segundo alumno)
notas[:, 2]     # toda la columna 2 (tercer examen)
```

Los `:` significan **"todo"** en esa dimensión. Así, `notas[:, 2]` se lee *"todas las filas, columna 2"*.

### 🎯 Tu misión
Guarda en `resultado` las notas del **segundo examen** (columna 1) de todos los alumnos.

<details><summary>💡 Pista</summary>

`notas[:, 1]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np05")

## 6. Recortar submatrices
`Intermedio` · **+20 XP**

Con **rebanadas** (*slicing*) `inicio:fin` puedes recortar trozos de un array. Como en Python, el `fin` no se incluye:

```python
notas[:2]          # primeras 2 filas
notas[1:4, 0:2]    # filas 1 a 3, columnas 0 y 1
notas[-2:]         # últimas 2 filas
```

> ⚠️ Un recorte es una **vista** de los mismos datos: si lo modificas, también cambia el original. Usa `.copy()` si quieres un array independiente.

### 🎯 Tu misión
Guarda en `resultado` las notas de los **3 primeros alumnos** en los **2 últimos exámenes**.

<details><summary>💡 Pista</summary>

`notas[:3, 1:] o notas[:3, -2:]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np06")

## 7. Filtrar con máscaras
`Intermedio` · **+20 XP**

Igual que en pandas, una comparación genera un array de **True/False** que sirve para filtrar:

```python
a = np.array([4, 9, 2, 7])
a > 5          # [False  True False  True]
a[a > 5]       # [9 7]
```

Para combinar condiciones se usan `&` (y) y `|` (o), **con paréntesis**: `a[(a > 2) & (a < 8)]`.

> 💡 `(a > 5).sum()` cuenta cuántos valores cumplen la condición.

### 🎯 Tu misión
Guarda en `resultado` las temperaturas **mayores que 22** grados.

<details><summary>💡 Pista</summary>

`temperaturas[temperaturas > 22]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np07")

---
# 📌 Cálculo vectorizado

## 8. Fórmulas sobre arrays
`Básico` · **+10 XP**

Cualquier fórmula matemática se aplica **elemento a elemento** a todo el array, sin bucles:

```python
km = np.array([5, 10, 21])
millas = km * 0.621        # [ 3.1  6.2 13.0]
```

También funcionan las funciones matemáticas de NumPy: `np.sqrt`, `np.log`, `np.exp`, `np.round`, `np.abs`…

> 💼 **En el trabajo:** convertir unidades, normalizar datos o aplicar fórmulas financieras sobre millones de valores en milisegundos.

### 🎯 Tu misión
Convierte `temperaturas` de Celsius a **Fahrenheit** con la fórmula `F = C * 9/5 + 32`. Guarda el array.

<details><summary>💡 Pista</summary>

`temperaturas * 9 / 5 + 32`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np08")

## 9. Agregar por filas o columnas
`Avanzado` · **+35 XP**

Las funciones de resumen (`sum`, `mean`, `max`, `min`, `std`) aceptan el parámetro **`axis`**:

- sin `axis`: resumen de **todo** el array;
- `axis=0`: **baja por las filas**, es decir, un resultado **por columna**;
- `axis=1`: **recorre las columnas**, es decir, un resultado **por fila**.

```python
m = np.array([[1, 2],
              [3, 4]])
m.sum()          # 10
m.sum(axis=0)    # [4 6]   (por columna)
m.sum(axis=1)    # [3 7]   (por fila)
```

> 💡 Truco para recordarlo: el `axis` que indicas es la dimensión que **desaparece**.

### 🎯 Tu misión
Calcula el **promedio de cada alumno** (cada fila de `notas`). Guarda el array de 5 promedios.

<details><summary>💡 Pista</summary>

`notas.mean(axis=1)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np09")

## 10. Broadcasting
`Avanzado` · **+35 XP**

El **broadcasting** permite operar arrays de formas distintas: NumPy "estira" el más pequeño para que encaje con el grande.

```python
notas.shape        # (5, 3)
bonus = np.array([1, 0, 0.5])      # shape (3,)
notas + bonus      # suma el bonus correspondiente a CADA columna
```

El vector `bonus` se aplica a las 5 filas: +1 al primer examen, +0 al segundo y +0.5 al tercero.

Después, `np.minimum(array, 10)` limita los valores para que ninguno pase de 10 (`np.clip(array, 0, 10)` limita por arriba y por abajo).

### 🎯 Tu misión
Suma a `notas` el bonus `[1, 0, 0.5]` (uno por examen) y limita las notas a un **máximo de 10** con `np.minimum`. Guarda la matriz.

<details><summary>💡 Pista</summary>

`np.minimum(notas + bonus, 10)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

bonus = np.array([1, 0, 0.5])
resultado = 

comprobar("np10")

## 11. Condicionales con np.where
`Intermedio` · **+20 XP**

`np.where(condición, valor_si_true, valor_si_false)` es un **if vectorizado**: evalúa la condición para cada elemento y elige un valor u otro.

```python
edades = np.array([15, 30, 17])
np.where(edades >= 18, "adulto", "menor")
# ['menor' 'adulto' 'menor']
```

Se usa muchísimo para crear columnas en pandas:

```python
df["tipo"] = np.where(df["precio"] > 100, "caro", "barato")
```

### 🎯 Tu misión
Con el promedio de cada alumno (`promedios`), crea un array que diga **"aprobado"** si el promedio es **≥ 7** y **"recupera"** si no.

<details><summary>💡 Pista</summary>

`np.where(promedios >= 7, 'aprobado', 'recupera')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

promedios = notas.mean(axis=1)
resultado = 

comprobar("np11")

---
# 📌 NumPy en la práctica

## 12. Números aleatorios reproducibles
`Intermedio` · **+20 XP**

Para simular datos, mezclar filas o separar datos de entrenamiento se necesitan **números aleatorios**. La forma moderna es crear un **generador** con una **semilla** (*seed*):

```python
rng = np.random.default_rng(42)
rng.integers(1, 7, size=5)     # 5 tiradas de dado (el 7 no se incluye)
rng.normal(0, 1, size=3)       # 3 valores de una distribución normal
rng.choice(["a", "b"], size=4) # elecciones al azar
```

Con la **misma semilla** siempre salen **los mismos números**. Eso hace que tus experimentos sean **reproducibles**.

> 💼 **En el trabajo:** en Machine Learning siempre se fija la semilla (`random_state=42`) para que otra persona pueda obtener exactamente tus resultados.

### 🎯 Tu misión
Crea un generador con semilla **0** y simula **10 tiradas de un dado** (enteros del 1 al 6). Guarda el array.

<details><summary>💡 Pista</summary>

`rng = np.random.default_rng(0); rng.integers(1, 7, size=10)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

rng = 
resultado = 

comprobar("np12")

## 13. Estandarizar datos
`Avanzado` · **+35 XP**

Estandarizar (calcular el **z-score**) transforma los datos para que tengan **media 0 y desviación estándar 1**:

```
z = (x - media) / desviación
```

Así ves cuántas desviaciones se aleja cada valor de la media: `z = 2` significa "muy por encima de lo normal".

```python
x = np.array([10, 20, 30])
(x - x.mean()) / x.std()      # [-1.22  0.    1.22]
```

> 💼 **En el trabajo:** muchos modelos de Machine Learning (KNN, regresión logística, redes neuronales) funcionan mucho mejor con los datos estandarizados. Lo verás de nuevo en el último módulo con `StandardScaler`.

### 🎯 Tu misión
Estandariza `temperaturas`: resta su media y divide por su desviación estándar (`.std()`). Guarda el array.

<details><summary>💡 Pista</summary>

`(temperaturas - temperaturas.mean()) / temperaturas.std()`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np13")

## 14. Percentiles
`Intermedio` · **+20 XP**

Un **percentil** indica el valor por debajo del cual queda cierto porcentaje de los datos:

- percentil 50 = **mediana** (la mitad de los datos está por debajo);
- percentil 90 = el 90% de los datos está por debajo.

```python
np.percentile(datos, 50)          # mediana
np.percentile(datos, [25, 75])    # primer y tercer cuartil
```

A diferencia de la media, los percentiles **no se deforman** con valores extremos. Por eso se usan para medir tiempos de respuesta ("el p95 es de 300 ms") o para detectar outliers.

> 💡 Puedes sacar un array de pandas con `df["col"].to_numpy()`, como se hizo con `precios`.

### 🎯 Tu misión
Calcula el **percentil 75** de `precios` y guárdalo en `resultado`.

<details><summary>💡 Pista</summary>

`np.percentile(precios, 75)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("np14")

## 15. Álgebra de matrices
`Pro` · **+50 XP**

El **producto matricial** (`@` o `np.dot`) es la base de la estadística y del Machine Learning: una regresión lineal predice con `X @ coeficientes`.

Para multiplicar `A @ B`, las columnas de A deben coincidir con las filas de B: `(5, 3) @ (3,)` da `(5,)`.

```python
ventas = np.array([[2, 1],
                   [0, 3]])          # 2 clientes x 2 productos (unidades)
precio = np.array([10, 4])          # precio de cada producto
ventas @ precio                      # [24 12]  -> gasto de cada cliente
```

> 💡 No lo confundas con `*`, que multiplica **elemento a elemento**.

### 🎯 Tu misión
Los exámenes pesan distinto: **30%, 30% y 40%**. Calcula la **nota final ponderada** de cada alumno con el producto matricial `notas @ pesos`.

<details><summary>💡 Pista</summary>

`notas @ pesos`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

pesos = np.array([0.3, 0.3, 0.4])
resultado = 

comprobar("np15")

---
# 🏁 ¡Módulo completado!
Vuelve a [PandasQuest](https://rosalesluciano.github.io/pandas-quest/) para sumar tus XP y seguir con el siguiente módulo. 🚀